In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys

ROOT = Path("/content/drive/MyDrive/TP-PSI-G8-main")

assert (ROOT / "src" / "pipeline.py").exists(), (
    "Revisá la ruta del repositorio actualizado."
)

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Proyecto:", ROOT)

Mounted at /content/drive
Proyecto: /content/drive/MyDrive/TP-PSI-G8-main


# Etapa 8 — Robustez

Se evalúa cómo cambia el desempeño del algoritmo al agregar
compresión JPEG, desenfoque y ruido a las imágenes de desarrollo.

Se comparan las reglas de fracción fija y valor fijo, manteniendo
los modelos sin reajustarlos a las degradaciones. También se
evalúa si los pasos de restauración mejoran el resultado.

In [2]:
import io
import joblib
import numpy as np
import pandas as pd
from PIL import Image

from src import config as C
from src.data import dev_ids, load_image, load_vessels, load_fov
from src.pipeline import feature_scores, load_dev_mask
from src.enhance import upscale
from src.fov import detect_fov_robust
from src.segment import segmentar_fraccion
from src.metrics import matthews, dice

ajustes = joblib.load(
    ROOT / "results" / "08_ajustes_cv.joblib"
)

assert ajustes["features"] == list(C.SEGMENT_FEATURES)
assert ajustes["balanced"] == C.SEGMENT_BALANCED_TRAINING
assert ajustes["factor"] == C.SEGMENT_FACTOR

fits = ajustes["fits"]
folds = ajustes["folds"]
ids = dev_ids()

## Robustez frente a compresión JPEG

Se compara la fotografía base —que ya tiene compresión JPEG—
con versiones recomprimidas a calidades 90, 70, 50 y 30.

Cada imagen se evalúa con los pesos y umbrales ajustados sin
su persona en el 05d. Se mantienen ambos umbrales sin reajustarlos
a las imágenes degradadas.

En este primer experimento no se aplica restauración adicional.

In [3]:
def recomprimir_jpeg(foto, calidad):
    """Recomprime en memoria, conservando las dimensiones."""
    archivo = io.BytesIO()
    Image.fromarray(foto).save(
        archivo, format="JPEG", quality=calidad, subsampling=2
    )
    archivo.seek(0)
    with Image.open(archivo) as imagen:
        return np.asarray(imagen.convert("RGB")).copy()


# Archivo propio de estos ajustes, para no mezclar corridas.
import hashlib

identificador = hashlib.sha256(
    (ROOT / "results" / "08_ajustes_cv.joblib").read_bytes()
).hexdigest()[:10]

salida = ROOT / "results" / f"08_jpeg_cv_{identificador}.csv"
calidades = [None, 90, 70, 50, 30]

previo = pd.read_csv(salida) if salida.exists() else pd.DataFrame()
filas = previo.to_dict("records")

for numero, imagen_id in enumerate(ids, start=1):
    foto = load_image(imagen_id)
    referencia = load_vessels(imagen_id)
    campo_ref = load_fov(imagen_id)
    ajuste = fits[int(folds.loc[imagen_id])]
    fraccion, valor = ajuste["combinación"]

    for calidad in calidades:
        nivel = "Base" if calidad is None else f"JPEG {calidad}"

        terminadas = {
            fila["Regla"] for fila in filas
            if fila["Imagen"] == imagen_id and fila["Nivel"] == nivel
        }
        if terminadas == {"Fracción fija", "Valor fijo"}:
            continue

        if calidad is None:
            mascaras = {
                "Fracción fija": load_dev_mask(
                    imagen_id, kind="combinación"
                ),
                "Valor fijo": load_dev_mask(
                    imagen_id, kind="combinación valor fijo"
                ),
            }
        else:
            degradada = recomprimir_jpeg(foto, calidad)

            probabilidades, _, _ = feature_scores(
                degradada, [ajuste["w"]]
            )
            puntuacion = upscale(
                probabilidades[0], referencia.shape
            )
            campo_auto = detect_fov_robust(degradada)

            mascaras = {
                "Fracción fija": segmentar_fraccion(
                    puntuacion, campo_auto, fraccion
                )[0],
                "Valor fijo": (puntuacion > valor) & campo_auto,
            }

        for regla, mascara in mascaras.items():
            if regla in terminadas:
                continue

            filas.append({
                "Imagen": imagen_id,
                "Nivel": nivel,
                "Regla": regla,
                "Matthews": matthews(
                    mascara, referencia, campo_ref
                ),
                "Dice": dice(mascara, referencia, campo_ref),
            })

        pd.DataFrame(filas).to_csv(salida, index=False)
        print(
            f"{numero}/{len(ids)} — {imagen_id}: {nivel}",
            flush=True
        )

tabla_jpeg = pd.DataFrame(filas)

display(
    tabla_jpeg.groupby(["Nivel", "Regla"])[["Matthews", "Dice"]]
    .mean()
    .reindex(
        pd.MultiIndex.from_product(
            [
                ["Base", "JPEG 90", "JPEG 70", "JPEG 50", "JPEG 30"],
                ["Fracción fija", "Valor fijo"],
            ],
            names=["Nivel", "Regla"],
        )
    )
    .round(4)
)

1/30 — 01_h: Base
1/30 — 01_h: JPEG 90
1/30 — 01_h: JPEG 70
1/30 — 01_h: JPEG 50
1/30 — 01_h: JPEG 30
2/30 — 02_h: Base
2/30 — 02_h: JPEG 90
2/30 — 02_h: JPEG 70
2/30 — 02_h: JPEG 50
2/30 — 02_h: JPEG 30
3/30 — 03_h: Base
3/30 — 03_h: JPEG 90
3/30 — 03_h: JPEG 70
3/30 — 03_h: JPEG 50
3/30 — 03_h: JPEG 30
4/30 — 04_h: Base
4/30 — 04_h: JPEG 90
4/30 — 04_h: JPEG 70
4/30 — 04_h: JPEG 50
4/30 — 04_h: JPEG 30
5/30 — 07_h: Base
5/30 — 07_h: JPEG 90
5/30 — 07_h: JPEG 70
5/30 — 07_h: JPEG 50
5/30 — 07_h: JPEG 30
6/30 — 08_h: Base
6/30 — 08_h: JPEG 90
6/30 — 08_h: JPEG 70
6/30 — 08_h: JPEG 50
6/30 — 08_h: JPEG 30
7/30 — 09_h: Base
7/30 — 09_h: JPEG 90
7/30 — 09_h: JPEG 70
7/30 — 09_h: JPEG 50
7/30 — 09_h: JPEG 30
8/30 — 10_h: Base
8/30 — 10_h: JPEG 90
8/30 — 10_h: JPEG 70
8/30 — 10_h: JPEG 50
8/30 — 10_h: JPEG 30
9/30 — 13_h: Base
9/30 — 13_h: JPEG 90
9/30 — 13_h: JPEG 70
9/30 — 13_h: JPEG 50
9/30 — 13_h: JPEG 30
10/30 — 14_h: Base
10/30 — 14_h: JPEG 90
10/30 — 14_h: JPEG 70
10/30 — 14_h: JPEG 

Matthews    Dice
Nivel   Regla                          
Base    Fracción fija    0.7255  0.7468
        Valor fijo       0.7302  0.7517
JPEG 90 Fracción fija    0.7244  0.7458
        Valor fijo       0.7289  0.7505
JPEG 70 Fracción fija    0.7178  0.7398
        Valor fijo       0.7228  0.7445
JPEG 50 Fracción fija    0.7106  0.7334
        Valor fijo       0.7155  0.7375
JPEG 30 Fracción fija    0.6942  0.7185
        Valor fijo       0.7007  0.7240

### Resultado frente a compresión JPEG

El rendimiento disminuyó al reducir la calidad JPEG. La calidad
90 produjo cambios pequeños; la mayor caída apareció en calidad 30.

El valor fijo obtuvo mayores medias de Matthews y Dice en todos
los niveles. En JPEG 30 alcanzó Matthews de 0,7007, frente a
0,6942 con fracción fija.

Estas medias todavía no determinan la elección del umbral:
los intervalos por personas se calcularán junto con las otras
degradaciones. No se aplicó restauración adicional en esta corrida.